# Gemini Notebook — Focused Angle + Unified Arabic + Line-by-Line Format

هذه النسخة تضبط المشاكل التي ظهرت في النتائج السابقة:

- **العربية تكون موحدة**: فصحى بسيطة فقط، بدون خلط عامية
- **اسم المنتج يبقى بالإنجليزي** داخل النص
- **فصل واضح جدًا بين الزوايا الثلاث**:
  - Product = يركز على المنتج نفسه فقط
  - Lifestyle = يركز على أسلوب الحياة فقط
  - Campaign = يركز على الحملة والـ CTA فقط
- **التنسيق يكون سطر تحت سطر**
- **كل سطر يبدأ بإيموجي**
- **الأفعال تكون موحدة** بصيغة محايدة/أمر موحد:
  - استمتع
  - جدد
  - اكتشف
  - تسوق
بدل صياغات مختلطة مثل:
- استمتعي / جددي / اكتشفي

## 1) تثبيت المكتبات

In [ ]:
!pip install -q -U google-genai pandas pydantic ipywidgets

## 2) إدخال Gemini API Key

In [ ]:
import os

# Add GEMINI_API_KEY in Google Colab Secrets or set it as an environment variable.
# Never paste a live API key into a notebook or commit it to GitHub.
try:
    from google.colab import userdata
    secret = userdata.get("GEMINI_API_KEY")
    if secret:
        os.environ["GEMINI_API_KEY"] = secret
except Exception:
    pass

if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError("Configure GEMINI_API_KEY securely before running this notebook.")


## 3) استيراد المكتبات

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List
import pandas as pd
import json
import time
from IPython.display import display, Markdown

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## 4) الموديلات المقترحة كـ fallback

In [ ]:
FALLBACK_MODELS = [
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-1.5-flash",
]

MODEL_NAME = FALLBACK_MODELS[0]

## 5) مثال بيانات منتج

In [ ]:
product = {
    "Product_Category": "Women Fashion",
    "Sub_Category": "Dress",
    "Product_Name": "Linen Summer Midi Dress",
    "Product_Description_EN": "A breathable linen midi dress with a relaxed silhouette, soft texture, and elegant cut for warm-weather styling.",
    "Material": "Linen",
    "Season": "Summer",
    "Fashion_Style": "Minimal Elegant",
    "Occasion": "Casual outing",
    "Gender": "Women",
    "Target_Audience": "Young women aged 20-35",
    "Campaign_Goal": "Increase product awareness and conversions",
    "Platform": "Instagram",
    "Tone": "Warm and stylish",
    "Emotion": "Confidence",
    "Call_to_Action": "Shop now",
    "USP": "Breathable premium linen with effortless elegance"
}

## 6) شخصيات المنصات

لاحظي أننا هنا أبقينا اللغة **فصحى بسيطة فقط** حتى على TikTok،
لكن مع اختلاف في:
- الطول
- السرعة
- الهوك
- الزاوية البيعية

In [ ]:
PLATFORM_PROFILES = {
    "Instagram": {
        "hashtags_count": 20,
        "register_level": "Simple polished Modern Standard Arabic only. No slang. No dialect. No mixed Arabic.",
        "hook_style": "Visual or aspirational opening.",
        "selling_angle": "Sell elegance, confidence, and effortless style.",
        "cta_style": "Soft CTA.",
        "text_length": "3 short lines. Each line 6 to 14 words.",
        "emoji_rule": "Place ONE emoji at the END of each line before the punctuation."
    },
    "TikTok": {
        "hashtags_count": 5,
        "register_level": "Simple modern standard Arabic only. No slang, no dialect, no colloquial words.",
        "hook_style": "Fast, catchy, high-attention opening.",
        "selling_angle": "Sell instant desire, easy styling, and trend energy.",
        "cta_style": "Quick direct CTA.",
        "text_length": "2 very short lines. Each line 4 to 10 words.",
        "emoji_rule": "Place ONE emoji at the END of each line before the punctuation."
    },
    "Facebook": {
        "hashtags_count": 6,
        "register_level": "Clear friendly Modern Standard Arabic only.",
        "hook_style": "Practical and relatable opening.",
        "selling_angle": "Sell comfort, daily use, and value.",
        "cta_style": "Direct warm CTA.",
        "text_length": "4 medium lines. Each line 7 to 16 words.",
        "emoji_rule": "Place ONE emoji at the END of each line before the punctuation."
    },
    "Website": {
        "hashtags_count": 5,
        "register_level": "Clean professional Modern Standard Arabic only.",
        "hook_style": "Benefit-led opening.",
        "selling_angle": "Sell clarity, quality, product value, and conversion.",
        "cta_style": "Clean conversion CTA.",
        "text_length": "2 medium lines. Each line 7 to 16 words.",
        "emoji_rule": "Place ONE emoji at the END of each line before the punctuation."
    },
    "X": {
        "hashtags_count": 4,
        "register_level": "Short modern standard Arabic only.",
        "hook_style": "Sharp direct opening.",
        "selling_angle": "Sell one core benefit only.",
        "cta_style": "Minimal CTA.",
        "text_length": "2 or 3 short lines. Each line 4 to 9 words.",
        "emoji_rule": "Place ONE emoji at the END of each line before the punctuation."
    },
    "LinkedIn": {
        "hashtags_count": 5,
        "register_level": "Professional refined Modern Standard Arabic only.",
        "hook_style": "Value-led opening.",
        "selling_angle": "Sell quality, thoughtful design, and brand value.",
        "cta_style": "Professional CTA.",
        "text_length": "3 medium lines. Each line 7 to 16 words.",
        "emoji_rule": "Place ONE emoji at the END of each line before the punctuation."
    }
}


## 7) تعريف Structured Output

In [ ]:
class PostBlock(BaseModel):
    text_ar: str = Field(description="Arabic marketing text only, formatted as separate lines")
    hashtags_en: List[str] = Field(description="English hashtags only")

class PlatformOutput(BaseModel):
    platform: str
    product_post: PostBlock
    lifestyle_post: PostBlock
    campaign_post: PostBlock

## 8) بناء البرومبت — نسخة مشددة جدًا

هذا الجزء يفرض:
- فصحى فقط
- لا خلط عامية
- اسم المنتج بالإنجليزي
- فصل واضح بين الزوايا
- 3 سطور
- كل سطر يبدأ بإيموجي

In [ ]:
def build_prompt(product_data: dict, platform_name: str) -> str:
    profile = PLATFORM_PROFILES[platform_name]
    count = profile["hashtags_count"]
    product_name = product_data.get("Product_Name", "the product")

    return f'''
You are a senior bilingual marketing copywriter.

Generate platform-specific Arabic marketing copy for ONE product.

GLOBAL LANGUAGE RULES:
- Write in Modern Standard Arabic only.
- Do NOT use dialect, slang, colloquial Arabic, or mixed register.
- Keep the Arabic unified and consistent.
- Use ONLY neutral (gender-neutral) command verb forms, such as:
  استمتع، جدد، اكتشف، تسوق
- STRICTLY FORBIDDEN: Do NOT use feminine verb forms under any circumstances, including:
  استمتعي، جددي، اكتشفي، تسوقي، انطلقي، جربي، احصلي
  This rule applies to ALL three posts (product, lifestyle, campaign).
- Keep the product name in English exactly as provided: {product_name}
- Do NOT translate the product name into Arabic.
- All hashtags must be in English only.
- Every hashtag MUST start with the # symbol. Example: #SummerDress, #Fashion
- NEVER return a hashtag without the # symbol.

GLOBAL FORMATTING RULES:
- Each post must contain exactly the number of lines specified in the platform profile below.
- Put each line on a new line, not in one paragraph.
- Do NOT start lines with emojis.
- Place ONE relevant emoji at the END of each line, BEFORE the punctuation mark (period or comma).
  Example: هذا فستان أنيق مناسب لكل المناسبات ✨.
  NOT: ✨ هذا فستان أنيق مناسب لكل المناسبات.
- Do NOT add extra blank lines between the text lines.
- Make the lines visually neat and easy to read.

SELECTED PLATFORM: {platform_name}

PLATFORM PROFILE:
- Register level: {profile["register_level"]}
- Hook style: {profile["hook_style"]}
- Selling angle: {profile["selling_angle"]}
- CTA style: {profile["cta_style"]}
- Preferred length: {profile["text_length"]}
- Emoji rule: Place ONE emoji at the END of each line before the punctuation.

ANGLE SEPARATION RULES:
1) product_post
- Focus ONLY on the product itself.
- Talk about the product name, material, design, quality, benefit, and USP.
- Do NOT focus on lifestyle scenes.
- Do NOT focus on campaign urgency except for a very light ending if needed.

2) lifestyle_post
- Focus ONLY on lifestyle, mood, occasion, season, audience fit, and daily look.
- Do NOT mention material, fabric, or detailed product specs unless absolutely necessary.
- Do NOT sound like product description.
- Make it feel like the product fits a certain way of living.

3) campaign_post
- Focus ONLY on campaign action.
- Emphasize launch, collection, offer, urgency, CTA, shopping action, or campaign objective.
- Do NOT focus on material details.
- Do NOT sound like a generic product description.
- This post must clearly feel like a campaign post.

HOOK RULES:
- The first line must act as the hook.
- The hook must match the platform style and the content angle.

HASHTAG RULES:
- Return EXACTLY {count} hashtags for EACH post.
- Every single hashtag MUST begin with # (e.g. #SummerDress not SummerDress).
- Keep hashtags relevant to the angle.
- No repeated hashtags within the same post.

PRODUCT DATA:
{json.dumps(product_data, ensure_ascii=False, indent=2)}
'''.strip()


## 9) دالة التوليد مع retry + fallback

In [ ]:
def generate_posts(product_data: dict, platform_name: str, model_name: str = MODEL_NAME):
    prompt = build_prompt(product_data, platform_name)
    last_error = None

    models_to_try = [model_name] + [m for m in FALLBACK_MODELS if m != model_name]

    for chosen_model in models_to_try:
        for attempt in range(3):
            try:
                response = client.models.generate_content(
                    model=chosen_model,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0.45,
                        response_mime_type="application/json",
                        response_schema=PlatformOutput,
                    ),
                )

                if getattr(response, "parsed", None) is not None:
                    parsed = response.parsed
                    if isinstance(parsed, BaseModel):
                        return fix_result(parsed.model_dump())
                    if isinstance(parsed, dict):
                        return fix_result(parsed)

                text = response.text
                if isinstance(text, str):
                    return fix_result(json.loads(text))

                raise ValueError("Could not parse model response.")

            except Exception as e:
                last_error = e
                msg = str(e)

                if "503" in msg or "UNAVAILABLE" in msg:
                    wait_time = 2 ** attempt
                    print(f"[Retry] {chosen_model} unavailable. Waiting {wait_time}s...")
                    time.sleep(wait_time)
                    continue

                raise

    raise last_error


## 9.5) دالة إصلاح الهاشتاجات والأفعال المؤنثة (Post-Processing)

هذه الدالة تضمن بشكل برمجي:
- كل هاشتاج يبدأ بـ `#` بدون استثناء
- لا توجد أفعال مؤنثة في النص العربي


In [ ]:
import re

# ── Hashtag fix ───────────────────────────────────────────────────────────
def fix_hashtags(hashtags: list) -> list:
    """
    Robust hashtag fixer:
    - Splits space-separated strings (model sometimes returns one big string)
    - Ensures every tag starts with exactly one #
    - Removes Arabic-only tokens the model sneaks in
    - Strips any non-alphanumeric characters
    """
    all_tags = []
    for item in hashtags:
        all_tags.extend(item.strip().split())

    fixed = []
    for tag in all_tags:
        tag = tag.strip().lstrip('#')
        if not tag:
            continue
        if not re.search(r'[A-Za-z]', tag):
            continue
        tag = re.sub(r'[\u0600-\u06FF]', '', tag)
        tag = re.sub(r'[^\w]', '', tag)
        tag = tag.strip('_')
        if tag:
            fixed.append('#' + tag)
    return fixed


# ── Neutral wording fix ───────────────────────────────────────────────────
# Convert feminine and plural audience wording into neutral singular wording.
VERB_TO_NEUTRAL = [
    # Feminine singular
    ('اكتشفي', 'اكتشف'),
    ('استمتعي', 'استمتع'),
    ('جددي', 'جدد'),
    ('تسوقي', 'تسوق'),
    ('انطلقي', 'انطلق'),
    ('جربي', 'جرب'),
    ('احصلي', 'احصل'),
    ('أضيفي', 'أضف'),
    ('أطلقي', 'أطلق'),
    ('اشتري', 'اشترِ'),
    ('احجزي', 'احجز'),
    ('تألقي', 'تألق'),
    ('اختاري', 'اختر'),
    ('احتفلي', 'احتفل'),
    ('عيشي', 'عش'),
    ('ارتدي', 'ارتدِ'),
    ('استكشفي', 'استكشف'),
    ('كوني', 'كن'),
    ('ابدئي', 'ابدأ'),
    ('حققي', 'حقق'),
    ('تميزي', 'تميز'),
    ('اشعري', 'اشعر'),
    ('انعمي', 'انعم'),
    ('استعدي', 'استعد'),
    ('اظهري', 'اظهر'),
    ('ابرزي', 'أبرز'),
    ('اشعلي', 'اشعل'),
    ('اكملي', 'أكمل'),

    # Plural imperative / audience forms
    ('اكتشفوا', 'اكتشف'),
    ('استمتعوا', 'استمتع'),
    ('جددوا', 'جدد'),
    ('تسوقوا', 'تسوق'),
    ('انطلقوا', 'انطلق'),
    ('جربوا', 'جرب'),
    ('احصلوا', 'احصل'),
    ('أضيفوا', 'أضف'),
    ('أطلقوا', 'أطلق'),
    ('اشتروا', 'اشترِ'),
    ('احتفلوا', 'احتفل'),
    ('تألقوا', 'تألق'),
    ('اختاروا', 'اختر'),
    ('عيشوا', 'عش'),
    ('استكشفوا', 'استكشف'),
    ('كونوا', 'كن'),
    ('ابدأوا', 'ابدأ'),
    ('حققوا', 'حقق'),
    ('تميزوا', 'تميز'),
    ('اشعروا', 'اشعر'),
    ('انعموا', 'انعم'),
    ('اكملوا', 'أكمل'),
    ('زوروا', 'زر'),
    ('انقروا', 'انقر'),
    ('تفوتوا', 'تفوت'),
    ('لا تفوتوا', 'لا تفوت'),
    ('ابحثوا', 'ابحث'),
    ('تبحثون', 'تبحث'),
    ('ابحثوا عن', 'ابحث عن'),
]

PHRASE_TO_NEUTRAL = [
    ('خياركم', 'خيارك'),
    ('إطلالتكم', 'إطلالتك'),
    ('إطلالاتكم', 'إطلالاتك'),
    ('ثقتكم', 'ثقتك'),
    ('ذوقكم', 'ذوقك'),
    ('أسلوبكم', 'أسلوبك'),
    ('مظهركم', 'مظهرك'),
    ('خزانة ملابسكم', 'خزانة ملابسك'),
    ('ملابسكم', 'ملابسك'),
    ('نزهاتكم', 'نزهاتك'),
    ('لقاءاتكم', 'لقاءاتك'),
    ('وقتكم', 'وقتك'),
    ('يومكم', 'يومك'),
    ('مع أصدقائكم', 'مع الأصدقاء'),
    ('صفحتنا', 'الصفحة'),
]

def fix_verbs(text: str) -> str:
    """Replace feminine and plural forms with neutral singular wording."""
    for wrong, neutral in VERB_TO_NEUTRAL:
        text = re.sub(wrong, neutral, text)
    for wrong, neutral in PHRASE_TO_NEUTRAL:
        text = re.sub(wrong, neutral, text)
    return text


# ── Apply both fixes to a full result dict ───────────────────────────────
def fix_result(result: dict) -> dict:
    """Apply hashtag and neutral-wording fixes to all posts in a result."""
    for section in ['product_post', 'lifestyle_post', 'campaign_post']:
        if section not in result:
            continue
        post = result[section]
        if 'hashtags_en' in post:
            post['hashtags_en'] = fix_hashtags(post['hashtags_en'])
        if 'text_ar' in post:
            post['text_ar'] = fix_verbs(post['text_ar'])
    return result

print('✅ Post-processing functions loaded.')


## 10) دوال عرض مرتبة

In [ ]:
def render_post_block(title: str, block: dict):
    # Apply hashtag fix here as final safety net before display
    safe_hashtags = fix_hashtags(block["hashtags_en"])
    hashtags = " ".join(safe_hashtags)

    # Escape # only for display so the first hashtag shows correctly in Markdown
    hashtags_display = hashtags.replace("#", "\\#")

    display(Markdown(f"""### {title}

**Text (AR):**

{block["text_ar"]}

Hashtags (EN):
{hashtags_display}
"""))

def render_result(result: dict):
    display(Markdown(f'# Platform: {result["platform"]}'))
    render_post_block("Product Post", result["product_post"])
    render_post_block("Lifestyle Post", result["lifestyle_post"])
    render_post_block("Campaign Post", result["campaign_post"])


## 11) تجربة على منصة واحدة

In [ ]:

result_instagram = generate_posts(product, "Instagram")
render_result(result_instagram)

## 12) التحقق من عدد الهاشتاجات

In [ ]:
def count_hashtags_check(result: dict):
    rows = []
    for section in ["product_post", "lifestyle_post", "campaign_post"]:
        rows.append({
            "platform": result["platform"],
            "section": section,
            "hashtags_count": len(result[section]["hashtags_en"])
        })
    return pd.DataFrame(rows)

count_hashtags_check(result_instagram)

## 13) مقارنة عدة منصات

In [ ]:
selected_platforms = ["Instagram", "TikTok", "Facebook", "Website"]

all_results = {}
for p in selected_platforms:
    try:
        all_results[p] = generate_posts(product, p)
    except Exception as e:
        all_results[p] = {
            "platform": p,
            "error": str(e)
        }

list(all_results.keys())

## 14) عرض النتائج

In [ ]:
for p, result in all_results.items():
    if "error" in result:
        print(f"{p} -> ERROR: {result['error']}")
    else:
        render_result(result)

## 15) عرض بروفايل منصة

In [ ]:
def show_platform_profile(platform_name):
    profile = PLATFORM_PROFILES[platform_name]
    text = f'''
# {platform_name} Profile

**Register level:** {profile["register_level"]}
**Hook style:** {profile["hook_style"]}
**Selling angle:** {profile["selling_angle"]}
**CTA style:** {profile["cta_style"]}
**Text length:** {profile["text_length"]}
**Emoji rule:** {profile["emoji_rule"]}
'''
    display(Markdown(text))

show_platform_profile("TikTok")

## 16) تشغيل لاحق على CSV

In [ ]:
# مثال لاحقًا:
# df = pd.read_csv("your_dataset.csv")
# sample_df = df.head(3)
#
# outputs = []
# for _, row in sample_df.iterrows():
#     try:
#         out = generate_posts(row.to_dict(), "Instagram")
#         outputs.append(out)
#     except Exception as e:
#         outputs.append({"platform": "Instagram", "error": str(e)})
#
# outputs